# Quickstart: распаковка данных и запуск Хоукс-модели

Ноутбук распаковывает дневную сетку активности из `*.csv.gz` и прогоняет главный
эксперимент работы — **Scaled-baseline Hawkes** (полураспады 1 и 3 дня) на целевой
величине `to_ord` (число заказов за день).

Запускать из корня репозитория с активированным окружением (см. README → «Установка»).

## 1. Распаковка данных

В репозитории лежит сжатый `dayuses_cohort_10000_seed42_daily_grid.csv.gz` (~13 МБ).
Распаковываем его рядом в `.csv`, если он ещё не распакован.

In [5]:
import gzip
import shutil
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent

data_dir = ROOT / "data" / "processed" / "orbitals"
csv_gz = data_dir / "dayuses_cohort_10000_seed42_daily_grid.csv.gz"
csv_path = data_dir / "dayuses_cohort_10000_seed42_daily_grid.csv"

if not csv_path.exists():
    with gzip.open(csv_gz, "rb") as src, open(csv_path, "wb") as dst:
        shutil.copyfileobj(src, dst)
    print(f"Распаковано: {csv_path}")
else:
    print(f"Уже распаковано: {csv_path}")

print(f"Размер: {csv_path.stat().st_size / 1e6:.0f} МБ")

Уже распаковано: /Users/amosh239/repo/mkn/diploma/data/processed/orbitals/dayuses_cohort_10000_seed42_daily_grid.csv
Размер: 215 МБ


## 2. Беглый взгляд на данные

In [6]:
import pandas as pd

df = pd.read_csv(csv_path, parse_dates=["event_date"])
print(f"строк: {len(df):,}  пользователей: {df['user_id'].nunique():,}")
print(f"окно: {df['event_date'].min().date()} .. {df['event_date'].max().date()}")
df.head(2)

строк: 2,869,319  пользователей: 10,000
окно: 2025-01-01 .. 2025-10-31


,user_id,event_date,search,cat,searches,has_search_to_cart,has_search_to_ord,has_cat_to_cart,has_cat_to_ord,search_to_cart,search_to_ord,cat_to_cart,cat_to_ord,to_cart,to_ord,gmv
0,59,2025-01-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,59,2025-01-02,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 3. Запуск Scaled-baseline Hawkes

Скрипт `scripts/compute/run_experimental_1_hawkes.py` со значениями по умолчанию
воспроизводит числа из работы. Артефакты (графики, таблицы, `summary.json`)
скрипт сам складывает в `diploma/reports/experimental_1_hawkes/` (папка создаётся
автоматически).

> **Для проверяющего.** Готовые отчёты всех экспериментов уже лежат в репозитории
> в `diploma/reports/`. Запуск этого скрипта пересоздаёт `experimental_1_hawkes/`. Числа в отчете при этом не меняются.

In [7]:
import subprocess
import sys

proc = subprocess.run(
    [sys.executable, "scripts/compute/run_experimental_1_hawkes.py"],
    cwd=ROOT,
    capture_output=True,
    text=True,
)

## 4. Сверка с диплом

В работе для этой модели на тесте приведены такие значения NLL.
Сравним их с тем, что только что насчитали — должно совпасть.

In [8]:
import json

summary = json.loads((ROOT / "diploma/reports/experimental_1_hawkes/summary.json").read_text())

expected = {
    "Personalized Poisson": 0.4096,
    "Scaled-baseline Hawkes": 0.3958,
}
actual = {
    "Personalized Poisson": summary["test_metrics_personalized_poisson"]["mean_poisson_nll"],
    "Scaled-baseline Hawkes": summary["test_metrics_hawkes"]["mean_poisson_nll"],
}

print(f"{'модель':<26}{'из диплома':>12}{'получено':>12}   совпало")
all_ok = True
for name in expected:
    exp, act = expected[name], actual[name]
    ok = abs(exp - act) < 5e-4
    all_ok &= ok
    print(f"{name:<26}{exp:>12.4f}{act:>12.4f}   {'✓' if ok else '✗'}")

модель                      из диплома    получено   совпало
Personalized Poisson            0.4096      0.4096   ✓
Scaled-baseline Hawkes          0.3958      0.3958   ✓
